# TrendPulse Configuration

Set dataset paths, countries, and analysis thresholds before loading data. Dataset paths are resolved from the repository root, so this notebook can also be run from a subfolder.

In [1]:
from datetime import date, datetime
from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "dataset").is_dir()
)
FILE_PATHS: list[Path] = [
    PROJECT_ROOT / "dataset/GBvideos.csv",
    PROJECT_ROOT / "dataset/INvideos.csv",
    PROJECT_ROOT / "dataset/USvideos.csv",
]
CATEGORIES_PATH: Path = PROJECT_ROOT / "dataset/categories.csv"
COUNTRIES: list[str] = ["GB", "IN", "US"]

MIN_VIEWS: int = 1_000_000
MIN_LIKES: int = 10_000
ENGAGEMENT_RATE_THRESHOLD: float = 0.05

RUN_STARTED_AT: datetime = datetime.now().astimezone()

In [2]:
print("=" * 60)
print(f"TrendPulse data load | started: {RUN_STARTED_AT:%Y-%m-%d %H:%M:%S %Z}")
print(f"Countries to load ({len(COUNTRIES)}): {', '.join(COUNTRIES)}")
print("=" * 60)

TrendPulse data load | started: 2026-10-27 14:11:10 IST
Countries to load (3): GB, IN, US


## Loading country data

Load each country CSV into records, checking that the expected video fields are present. The optional `max_rows` argument can limit a load for quick testing.

In [3]:
REQUIRED_VIDEO_COLUMNS: set[str] = {
    "video_id",
    "trending_date",
    "title",
    "channel_title",
    "category_id",
    "publish_time",
    "tags",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "thumbnail_link",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "description",
}


def load_country(path, country, max_rows=None):
    df = pd.read_csv(path, nrows=max_rows)
    missing_columns = REQUIRED_VIDEO_COLUMNS.difference(df.columns)
    if missing_columns:
        raise ValueError(
            f"{path} is missing required columns: {', '.join(sorted(missing_columns))}"
        )

    df["country"] = country
    records = df.to_dict("records")
    print(f"Loaded {len(records):,} records for {country}.")
    return records



# Build Category Names Dictionary

In [4]:
categories_df = pd.read_csv(CATEGORIES_PATH)
required_category_columns = {"category_id", "category_name"}
missing_category_columns = required_category_columns.difference(categories_df.columns)
if missing_category_columns:
    raise ValueError(
        f"{CATEGORIES_PATH} is missing required columns: "
        f"{', '.join(sorted(missing_category_columns))}"
    )

category_names: dict[int, str] = categories_df.set_index("category_id")["category_name"].to_dict()
category_names

{1: 'Film & Animation',
 2: 'Autos & Vehicles',
 10: 'Music',
 15: 'Pets & Animals',
 17: 'Sports',
 19: 'Travel & Events',
 20: 'Gaming',
 22: 'People & Blogs',
 23: 'Comedy',
 24: 'Entertainment',
 25: 'News & Politics',
 26: 'Howto & Style',
 27: 'Education',
 28: 'Science & Technology',
 29: 'Nonprofits & Activism',
 30: 'Movies',
 43: 'Shows'}

In [5]:
videos_by_country = {
    country: load_country(path, country) for path, country in zip(FILE_PATHS, COUNTRIES)
}
record_counts = {
    country: len(records) for country, records in videos_by_country.items()
}
record_counts

Loaded 15,075 records for GB.
Loaded 15,075 records for IN.
Loaded 16,080 records for US.


{'GB': 15075, 'IN': 15075, 'US': 16080}

## Converting and cleaning records

Convert metric fields to integers, CSV flags to booleans, `trending_date` from `yy.dd.mm` to `date`, and `publish_time` to an ISO datetime. Keep every record: when a value cannot be parsed, substitute `0` for numeric fields, `False` for flags, or `None` for dates. This avoids losing otherwise useful rows while making missing or invalid values explicit. The failure counts below report substitutions by field.

In [6]:
numeric_fields = ("views", "likes", "dislikes", "comment_count")
flag_fields = (
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
)
#Initialize parse_failure_counts dictionary with zero counts for each field
parse_failure_counts = {
    field: 0
    for field in (*numeric_fields, *flag_fields, "trending_date", "publish_time")
}

#for country, records in videos_by_country.items():
 #   print(country, records[:2])
    
#Perform type conversion and error handling for each record in videos_by_country
for country_records in videos_by_country.values():
    for record in country_records:
        #Numeric fields
        for field in numeric_fields:
            try:
                record[field] = int(record[field])
            except (TypeError, ValueError, OverflowError):
                record[field] = 0
                parse_failure_counts[field] += 1

        #Flag fields
        for field in flag_fields:
            try:
                value = record[field]
                if isinstance(value, bool):
                    record[field] = value
                elif str(value).strip().lower() in {"true", "1"}:
                    record[field] = True
                elif str(value).strip().lower() in {"false", "0"}:
                    record[field] = False
                else:
                    raise ValueError(f"Invalid boolean value: {value!r}")
            except (TypeError, ValueError):
                record[field] = False
                parse_failure_counts[field] += 1

        #Trending date field
        try:
            value = record["trending_date"]
            if isinstance(value, datetime):
                record["trending_date"] = value.date()
            elif not isinstance(value, date):
                record["trending_date"] = datetime.strptime(
                    str(value).strip(), "%y.%d.%m"
                ).date()
        except (TypeError, ValueError, OverflowError):
            record["trending_date"] = None
            parse_failure_counts["trending_date"] += 1

        #Publish time field
        try:
            value = record["publish_time"]
            if not isinstance(value, datetime):
                record["publish_time"] = datetime.fromisoformat(
                    str(value).strip().replace("Z", "+00:00")
                )
        except (TypeError, ValueError, OverflowError):
            record["publish_time"] = None
            parse_failure_counts["publish_time"] += 1

sample_record = next(iter(videos_by_country.values()))[0]
print("Sample converted fields:")
print({field: sample_record[field] for field in (*numeric_fields, *flag_fields, "trending_date", "publish_time")})
print("Parse failures by field (substituted values):")
parse_failure_counts

Sample converted fields:
{'views': 84344, 'likes': 2155, 'dislikes': 163, 'comment_count': 475, 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'trending_date': datetime.date(2017, 11, 14), 'publish_time': datetime.datetime(2017, 10, 18, 7, 42, 2, tzinfo=datetime.timezone.utc)}
Parse failures by field (substituted values):


{'views': 0,
 'likes': 12,
 'dislikes': 0,
 'comment_count': 36,
 'comments_disabled': 0,
 'ratings_disabled': 0,
 'video_error_or_removed': 0,
 'trending_date': 0,
 'publish_time': 0}

## Core data structures

- **Lists of dictionaries** hold the cleaned records: lists preserve the records and their order, while dictionaries let us access each video's fields by name. Both are mutable, which is useful because the cleaning step updates record values.
- **Tuples** hold each video's `(video_id, trending_date, views, likes)` snapshot. A tuple is immutable, which suits a fixed group of values that should not be accidentally changed. The original dictionaries remain available for other analysis.
- **Sets** hold unique video IDs per country. Sets are mutable, automatically remove duplicates, and support intersection and difference to compare countries.
- **Dictionaries** count channel appearances and store the category summary. They are mutable key-value mappings, useful for looking up counts by channel, country, and category.

Python lists, dictionaries, and sets are mutable; tuples are immutable.

In [7]:
# Cleaned records
cleaned_records: list[dict[str, object]] = [
    record
    for country_records in videos_by_country.values()
    for record in country_records
]

#Create tupples applicable metrics for each record
video_metric_tuples: list[tuple[str, date | None, int, int]] = [
    (record["video_id"], record["trending_date"], record["views"], record["likes"])
    for record in cleaned_records
]
print(f"Cleaned records: {len(cleaned_records):,}; metric tuples: {len(video_metric_tuples):,}")
print(f"First metric tuple: {video_metric_tuples[0]}")

# Identify unique video IDs in each country
unique_video_ids = {
    country: {record["video_id"] for record in records}
    for country, records in videos_by_country.items()
}

# Find video IDs shared between countries
shared_between_countries = {
    "GB & IN": unique_video_ids["GB"] & unique_video_ids["IN"],
    "GB & US": unique_video_ids["GB"] & unique_video_ids["US"],
    "IN & US": unique_video_ids["IN"] & unique_video_ids["US"],
}
shared_video_ids = set().union(*shared_between_countries.values())
unique_video_ids_by_country = {
    country: country_ids
    - set().union(
        *(other_ids for other_country, other_ids in unique_video_ids.items()
          if other_country != country)
    )
    for country, country_ids in unique_video_ids.items()
}
print("Unique video IDs by country:", {country: len(ids) for country, ids in unique_video_ids.items()})
print("Shared IDs by country pair:", {pair: len(ids) for pair, ids in shared_between_countries.items()})
print(f"Video IDs appearing in multiple countries: {len(shared_video_ids):,}")
print("Video IDs unique to each country:", {country: len(ids) for country, ids in unique_video_ids_by_country.items()})

channel_appearances: dict[str, int] = {}
for record in cleaned_records:
    channel = record["channel_title"]
    channel_appearances[channel] = channel_appearances.get(channel, 0) + 1

top_15_channels = sorted(
    channel_appearances.items(), key=lambda item: item[1], reverse=True
)[:15]
print("Top 15 channels by trending appearances:")
for rank, (channel, appearances) in enumerate(top_15_channels, start=1):
    print(f"{rank:>2}. {channel}: {appearances:,}")

category_summary: dict[str, dict[str, int]] = {}
for country, records in videos_by_country.items():
    category_summary[country] = {}
    for record in records:
        category_id = record["category_id"]
        category_name = category_names.get(
            category_id, f"Unknown category {category_id}"
        )
        category_summary[country][category_name] = (
            category_summary[country].get(category_name, 0) + 1
        )

print("Category summary examples:")
for country in COUNTRIES:
    leading_categories = sorted(
        category_summary[country].items(), key=lambda item: item[1], reverse=True
    )[:3]
    print(f"{country}: {leading_categories}")

Cleaned records: 46,230; metric tuples: 46,230
First metric tuple: ('u_nKn025TZS', datetime.date(2017, 11, 14), 84344, 2155)
Unique video IDs by country: {'GB': 4113, 'IN': 5878, 'US': 5068}
Shared IDs by country pair: {'GB & IN': 126, 'GB & US': 134, 'IN & US': 120}
Video IDs appearing in multiple countries: 340
Video IDs unique to each country: {'GB': 3873, 'IN': 5652, 'US': 4834}
Top 15 channels by trending appearances:
 1. QuietChallenge75: 2,464
 2. OfficialReview41: 1,586
 3. UltimateUnboxing65: 1,433
 4. InsaneReaction46: 1,233
 5. Brand NewChallenge45: 1,175
 6. CrazyTrailer91: 1,018
 7. FinalTrailer17: 911
 8. UltimateSpeech20: 770
 9. Brand NewSecret51: 684
10. WildHighlights7: 682
11. SurprisingHighlights41: 650
12. FinalBreakdown11: 643
13. CrazyRecipe67: 603
14. QuietInterview9: 570
15. UltimateVlog96: 565
Category summary examples:
GB: [('Music', 5013), ('Entertainment', 2844), ('People & Blogs', 1299)]
IN: [('Entertainment', 5117), ('Music', 2298), ('News & Politics', 11

## Variable scope

A variable defined outside a function is in the module (global) scope, so a function can read it. A variable assigned inside a function is local to that function and cannot be accessed by its name from outside. Returning a local value lets the caller use that value without making the local variable itself global.

In [8]:
global_var = "Defined outside the function."


def demonstrate_scope():
    local_var = "Local variable defined inside the function."
    print(f"Inside the function, I can read the outside variable: {global_var}")
    return local_var


returned_local_value = demonstrate_scope()
print(f"The function returned the local value: {returned_local_value}")

try:
    print(local_var)
    
except NameError as error:
    print(f"Outside the function, the local name is unavailable: {error}")

Inside the function, I can read the outside variable: Defined outside the function.
The function returned the local value: Local variable defined inside the function.
Outside the function, the local name is unavailable: name 'local_var' is not defined


## Loops over strings and sequences

`title_features` uses a `for` loop to examine punctuation characters and `range()` with index access to inspect words. It counts uppercase words after stripping surrounding punctuation, counts `!` and `?` characters, and reports the number of whitespace-separated words. The search below uses a `while` loop whose index is bounded by the record count: `continue` skips records below the view threshold, and `break` stops at the first qualifying record. If no record qualifies, the loop ends safely.

In [9]:
import string


def title_features(title: str) -> dict[str, int]:
    uppercase_words = 0
    exclamation_marks = 0
    question_marks = 0
    # for loop with if-elif statements to count exclamation and question marks
    for character in title:
        if character == "!":
            exclamation_marks += 1
        elif character == "?":
            question_marks += 1

    words = title.split()
    #for loop with range(len(words)) to count uppercase words
    for index in range(len(words)):
        word = words[index].strip(string.punctuation)
        if word and any(character.isalpha() for character in word) and word.isupper():
            uppercase_words += 1

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": exclamation_marks,
        "question_marks": question_marks,
        "word_count": len(words),
    }


example_title = "WOW! New VIDEO?"
print(f"Features for {example_title!r}: {title_features(example_title)}")

view_threshold = 1_000_000
first_high_view_video = None
record_index = 0
#while loop to find the first video with at least view_threshold views
while record_index < len(cleaned_records):
    candidate = cleaned_records[record_index]
    record_index += 1
    if candidate["views"] < view_threshold:
        continue
    first_high_view_video = candidate
    break

if first_high_view_video is None:
    print(f"No video found with at least {view_threshold:,} views.")
else:
    print(f"First video with at least {view_threshold:,} views: {first_high_view_video['title']}")
    print(f"Title features: {title_features(first_high_view_video['title'])}")

Features for 'WOW! New VIDEO?': {'uppercase_words': 2, 'exclamation_marks': 1, 'question_marks': 1, 'word_count': 3}
First video with at least 1,000,000 views: Lena Ortiz - Phones (Official Video)
Title features: {'uppercase_words': 0, 'exclamation_marks': 0, 'question_marks': 0, 'word_count': 6}


## Notebook discipline

This notebook uses clear headings, is intended to be run with **Restart and Run All**, and avoids hidden state by defining configuration before use.